<a href="https://colab.research.google.com/github/Fusingchart/voxelmorph/blob/main/Voxelmorphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
from __future__ import annotations

import math
import os
import random
import subprocess
import time
from collections import deque
from typing import List, Optional, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.functional as nnf
from torch.distributions.normal import Normal
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [69]:
GCS_BUCKET_NAME = "rayans_aimi_group_four_bucket"

DRIVE_ZIP_PATH = (
    "/content/drive/Shareddrives/Stanford AIMI Group 4/pediatric_echo_avi.zip"
)

DATA_ROOT: Optional[str] = None

RUN_COLAB_SETUP = True

DEMO = False  # Set to False to use real data

EPOCHS = 10  # Increased epochs for more substantial training
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
LAMBDA_GRAD = 0.5
IMAGE_SIZE = 128
NUM_WORKERS = 2 # Increased workers for faster data loading
MAX_VIDEOS = 50 # Increased to use more real data
SEED = 0

SAVE_PATH = "/content/vxm_echo.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [44]:
def _videos_dir(data_root: str) -> str:
    return os.path.join(data_root, "A4C", "Videos")


def _has_any_video(videos_dir: str) -> bool:
    if not os.path.isdir(videos_dir):
        return False
    for f in os.listdir(videos_dir):
        if f.lower().endswith((".avi", ".mp4", ".mov")):
            return True
    return False


def _find_parent_with_a4c_videos(start: str, max_depth: int) -> Optional[str]:
    """If A4C/Videos is nested (e.g. under an extra folder), find the parent of A4C."""
    queue: deque[Tuple[str, int]] = deque([(start, 0)])
    seen = set()
    while queue:
        path, depth = queue.popleft()
        if path in seen or len(path) > 512:
            continue
        seen.add(path)
        vd = _videos_dir(path)
        if _has_any_video(vd):
            return path
        if depth >= max_depth:
            continue
        try:
            for name in os.listdir(path):
                sub = os.path.join(path, name)
                if os.path.isdir(sub):
                    queue.append((sub, depth + 1))
        except OSError:
            pass
    return None


def discover_data_root() -> Optional[str]:
    """Resolve where EchoNet videos live (gcsfuse + Drive unzip, per dataloading.py)."""
    env = os.environ.get("ECHO_DATA_ROOT")
    if env:
        if _has_any_video(_videos_dir(env)):
            return os.path.normpath(env)
    if DATA_ROOT is not None:
        if _has_any_video(_videos_dir(DATA_ROOT)):
            return os.path.normpath(DATA_ROOT)
    for c in ("/mnt/data", "/content/local_data"):
        if _has_any_video(_videos_dir(c)):
            return c
    if os.path.isdir("/content/local_data"):
        nested = _find_parent_with_a4c_videos("/content/local_data", max_depth=6)
        if nested:
            return nested
    return None

In [45]:
def colab_setup_from_dataloading() -> None:
    """
    Same steps as dataloading.py: Drive → unzip → GCS auth → gcsfuse → /mnt/data.
    Run once per Colab session (or set RUN_COLAB_SETUP = True before train()).
    """
    try:
        from google.colab import auth
        from google.colab import drive
    except ImportError as e:
        raise RuntimeError("colab_setup_from_dataloading() only works in Google Colab.") from e

    print("Mounting Google Drive…")
    drive.mount("/content/drive")

    os.makedirs("/content/local_data", exist_ok=True)
    if os.path.isfile(DRIVE_ZIP_PATH):
        print("Unzipping pediatric_echo_avi.zip to /content/local_data …")
        subprocess.run(
            ["unzip", "-q", DRIVE_ZIP_PATH, "-d", "/content/local_data"],
            check=False,
        )
    else:
        print(f"Skipping unzip (zip not found): {DRIVE_ZIP_PATH}")

    print("Authenticate for GCS (bucket access)…")
    auth.authenticate_user()

    print("Installing gcsfuse…")
    subprocess.run(
        'echo "deb http://packages.cloud.google.com/apt gcsfuse-buster main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list',
        shell=True,
        check=True,
    )
    subprocess.run(
        "curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -",
        shell=True,
        check=True,
    )
    subprocess.run(["apt-get", "-q", "update"], check=True)
    subprocess.run(["apt-get", "-q", "-y", "install", "gcsfuse"], check=True)

    os.makedirs("/mnt/data", exist_ok=True)
    print(f"Mounting GCS bucket {GCS_BUCKET_NAME} to /mnt/data …")
    subprocess.run(
        ["gcsfuse", "--implicit-dirs", GCS_BUCKET_NAME, "/mnt/data"],
        check=True,
    )
    print("✅ Setup complete. Listing /mnt/data:")
    subprocess.run(["ls", "-l", "/mnt/data"], check=False)

In [46]:
def default_unet_features():
    return [
        [16, 32, 32, 32],
        [32, 32, 32, 32, 32, 16, 16],
    ]

In [47]:
class SpatialTransformer(nn.Module):
    def __init__(self, size, mode="bilinear"):
        super().__init__()
        self.mode = mode
        vectors = [torch.arange(0, s) for s in size]
        try:
            grids = torch.meshgrid(*vectors, indexing="ij")
        except TypeError:
            grids = torch.meshgrid(*vectors)
        grid = torch.stack(grids)
        grid = torch.unsqueeze(grid, 0)
        grid = grid.type(torch.FloatTensor)
        self.register_buffer("grid", grid)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        if len(shape) == 2:
            new_locs = new_locs.permute(0, 2, 3, 1)
            new_locs = new_locs[..., [1, 0]]
        elif len(shape) == 3:
            new_locs = new_locs.permute(0, 2, 3, 4, 1)
            new_locs = new_locs[..., [2, 1, 0]]
        return nnf.grid_sample(src, new_locs, align_corners=True, mode=self.mode)

In [48]:
class VecInt(nn.Module):
    def __init__(self, inshape, nsteps):
        super().__init__()
        assert nsteps >= 0, "nsteps should be >= 0, found: %d" % nsteps
        self.nsteps = nsteps
        self.scale = 1.0 / (2**self.nsteps)
        self.transformer = SpatialTransformer(inshape)

    def forward(self, vec):
        vec = vec * self.scale
        for _ in range(self.nsteps):
            vec = vec + self.transformer(vec, vec)
        return vec

In [49]:
class ResizeTransform(nn.Module):
    def __init__(self, vel_resize, ndims):
        super().__init__()
        self.factor = 1.0 / vel_resize
        self.mode = "linear"
        if ndims == 2:
            self.mode = "bi" + self.mode
        elif ndims == 3:
            self.mode = "tri" + self.mode

    def forward(self, x):
        if self.factor < 1:
            x = nnf.interpolate(x, align_corners=True, scale_factor=self.factor, mode=self.mode)
            x = self.factor * x
        elif self.factor > 1:
            x = self.factor * x
            x = nnf.interpolate(x, align_corners=True, scale_factor=self.factor, mode=self.mode)
        return x

In [50]:
class ConvBlock(nn.Module):
    def __init__(self, ndims, in_channels, out_channels, stride=1):
        super().__init__()
        Conv = getattr(nn, "Conv%dd" % ndims)
        self.main = Conv(in_channels, out_channels, 3, stride, 1)
        self.activation = nn.LeakyReLU(0.2)

    def forward(self, x):
        return self.activation(self.main(x))

In [51]:
class Unet(nn.Module):
    def __init__(
        self,
        inshape=None,
        infeats=None,
        nb_features=None,
        nb_levels=None,
        max_pool=2,
        feat_mult=1,
        nb_conv_per_level=1,
        half_res=False,
    ):
        super().__init__()
        ndims = len(inshape)
        assert ndims in [1, 2, 3], "ndims should be one of 1, 2, or 3. found: %d" % ndims
        self.half_res = half_res
        if nb_features is None:
            nb_features = default_unet_features()
        if isinstance(nb_features, int):
            if nb_levels is None:
                raise ValueError("must provide unet nb_levels if nb_features is an integer")
            feats = np.round(nb_features * feat_mult ** np.arange(nb_levels)).astype(int)
            nb_features = [
                np.repeat(feats[:-1], nb_conv_per_level),
                np.repeat(np.flip(feats), nb_conv_per_level),
            ]
        elif nb_levels is not None:
            raise ValueError("cannot use nb_levels if nb_features is not an integer")
        enc_nf, dec_nf = nb_features
        nb_dec_convs = len(enc_nf)
        final_convs = dec_nf[nb_dec_convs:]
        dec_nf = dec_nf[:nb_dec_convs]
        self.nb_levels = int(nb_dec_convs / nb_conv_per_level) + 1
        if isinstance(max_pool, int):
            max_pool = [max_pool] * self.nb_levels
        MaxPooling = getattr(nn, "MaxPool%dd" % ndims)
        self.pooling = [MaxPooling(s) for s in max_pool]
        self.upsampling = [nn.Upsample(scale_factor=s, mode="nearest") for s in max_pool]
        prev_nf = infeats
        encoder_nfs = [prev_nf]
        self.encoder = nn.ModuleList()
        for level in range(self.nb_levels - 1):
            convs = nn.ModuleList()
            for conv in range(nb_conv_per_level):
                nf = enc_nf[level * nb_conv_per_level + conv]
                convs.append(ConvBlock(ndims, prev_nf, nf))
                prev_nf = nf
            self.encoder.append(convs)
            encoder_nfs.append(prev_nf)
        encoder_nfs = np.flip(encoder_nfs)
        self.decoder = nn.ModuleList()
        for level in range(self.nb_levels - 1):
            convs = nn.ModuleList()
            for conv in range(nb_conv_per_level):
                nf = dec_nf[level * nb_conv_per_level + conv]
                convs.append(ConvBlock(ndims, prev_nf, nf))
                prev_nf = nf
            self.decoder.append(convs)
            if not half_res or level < (self.nb_levels - 2):
                prev_nf += encoder_nfs[level]
        self.remaining = nn.ModuleList()
        for _, nf in enumerate(final_convs):
            self.remaining.append(ConvBlock(ndims, prev_nf, nf))
            prev_nf = nf
        self.final_nf = prev_nf

    def forward(self, x):
        x_history = [x]
        for level, convs in enumerate(self.encoder):
            for conv in convs:
                x = conv(x)
            x_history.append(x)
            x = self.pooling[level](x)
        for level, convs in enumerate(self.decoder):
            for conv in convs:
                x = conv(x)
            if not self.half_res or level < (self.nb_levels - 2):
                x = self.upsampling[level](x)
                x = torch.cat([x, x_history.pop()], dim=1)
        for conv in self.remaining:
            x = conv(x)
        return x

In [52]:
class VxmDense(nn.Module):
    def __init__(
        self,
        inshape,
        nb_unet_features=None,
        nb_unet_levels=None,
        unet_feat_mult=1,
        nb_unet_conv_per_level=1,
        int_steps=7,
        int_downsize=2,
        bidir=False,
        use_probs=False,
        src_feats=1,
        trg_feats=1,
        unet_half_res=False,
    ):
        super().__init__()
        ndims = len(inshape)
        assert ndims in [1, 2, 3], "ndims should be one of 1, 2, or 3. found: %d" % ndims
        self.bidir = bidir
        self.unet_model = Unet(
            inshape,
            infeats=(src_feats + trg_feats),
            nb_features=nb_unet_features,
            nb_levels=nb_unet_levels,
            feat_mult=unet_feat_mult,
            nb_conv_per_level=nb_unet_conv_per_level,
            half_res=unet_half_res,
        )
        Conv = getattr(nn, "Conv%dd" % ndims)
        self.flow = Conv(self.unet_model.final_nf, ndims, kernel_size=3, padding=1)
        self.flow.weight = nn.Parameter(Normal(0, 1e-5).sample(self.flow.weight.shape))
        self.flow.bias = nn.Parameter(torch.zeros(self.flow.bias.shape))
        if use_probs:
            raise NotImplementedError("set use_probs to False for PyTorch")
        if not unet_half_res and int_steps > 0 and int_downsize > 1:
            self.resize = ResizeTransform(int_downsize, ndims)
        else:
            self.resize = None
        if int_steps > 0 and int_downsize > 1:
            self.fullsize = ResizeTransform(1 / int_downsize, ndims)
        else:
            self.fullsize = None
        down_shape = [int(dim / int_downsize) for dim in inshape]
        self.integrate = VecInt(down_shape, int_steps) if int_steps > 0 else None
        self.transformer = SpatialTransformer(inshape)

    def forward(self, source, target, registration=False):
        x = torch.cat([source, target], dim=1)
        x = self.unet_model(x)
        flow_field = self.flow(x)
        pos_flow = flow_field
        if self.resize:
            pos_flow = self.resize(pos_flow)
        preint_flow = pos_flow
        neg_flow = -pos_flow if self.bidir else None
        if self.integrate:
            pos_flow = self.integrate(pos_flow)
            neg_flow = self.integrate(neg_flow) if self.bidir else None
            if self.fullsize:
                pos_flow = self.fullsize(pos_flow)
                neg_flow = self.fullsize(neg_flow) if self.bidir else None
        y_source = self.transformer(source, pos_flow)
        y_target = self.transformer(target, neg_flow) if self.bidir else None
        if not registration:
            if self.bidir:
                return y_source, y_target, preint_flow
            return y_source, preint_flow
        return y_source, pos_flow

In [53]:
class NCC:
    def __init__(self, win=None):
        self.win = win

    def loss(self, y_true, y_pred):
        Ii = y_true
        Ji = y_pred
        ndims = len(list(Ii.size())) - 2
        assert ndims in [1, 2, 3], "volumes should be 1 to 3 dimensions. found: %d" % ndims
        win = [9] * ndims if self.win is None else self.win
        sum_filt = torch.ones([1, 1, *win], device=Ii.device, dtype=Ii.dtype)
        pad_no = math.floor(win[0] / 2)
        if ndims == 1:
            stride = (1,)
            padding = (pad_no,)
        elif ndims == 2:
            stride = (1, 1)
            padding = (pad_no, pad_no)
        else:
            stride = (1, 1, 1)
            padding = (pad_no, pad_no, pad_no)
        conv_fn = getattr(F, "conv%dd" % ndims)
        I2 = Ii * Ii
        J2 = Ji * Ji
        IJ = Ii * Ji
        I_sum = conv_fn(Ii, sum_filt, stride=stride, padding=padding)
        J_sum = conv_fn(Ji, sum_filt, stride=stride, padding=padding)
        I2_sum = conv_fn(I2, sum_filt, stride=stride, padding=padding)
        J2_sum = conv_fn(J2, sum_filt, stride=stride, padding=padding)
        IJ_sum = conv_fn(IJ, sum_filt, stride=stride, padding=padding)
        win_size = np.prod(win)
        u_I = I_sum / win_size
        u_J = J_sum / win_size
        cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
        I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
        J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size
        cc = cross * cross / (I_var * J_var + 1e-5)
        return -torch.mean(cc)

In [54]:
class Grad2D:
    def __init__(self, penalty="l1"):
        self.penalty = penalty

    def loss(self, flow):
        dy = torch.abs(flow[:, :, 1:, :] - flow[:, :, :-1, :])
        dx = torch.abs(flow[:, :, :, 1:] - flow[:, :, :, :-1])
        if self.penalty == "l2":
            dy = dy * dy
            dx = dx * dx
        return torch.mean(dy) + torch.mean(dx)

In [55]:
def _resize_gray(frame_bgr: np.ndarray, hw: Tuple[int, int]) -> np.ndarray:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.resize(gray, (hw[1], hw[0]), interpolation=cv2.INTER_AREA)


def _read_frame(path: str, index: int, hw: Tuple[int, int]) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    try:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ret, frame = cap.read()
        if not ret:
            return None
        return _resize_gray(frame, hw).astype(np.float32) / 255.0
    finally:
        cap.release()


def list_echo_videos(data_root: str) -> List[str]:
    videos_dir = os.path.join(data_root, "A4C", "Videos")
    if not os.path.isdir(videos_dir):
        return []
    names = sorted(
        f for f in os.listdir(videos_dir) if f.lower().endswith((".avi", ".mp4", ".mov"))
    )
    return [os.path.join(videos_dir, f) for f in names]

In [65]:
class EchoFramePairDataset(Dataset):
    def __init__(
        self,
        data_root: str,
        image_hw: Tuple[int, int] = (128, 128),
        seed: int = 0,
        max_videos: Optional[int] = None,
        augment: bool = False,
    ):
        super().__init__()
        self.data_root = data_root
        self.image_hw = image_hw
        self._rng = random.Random(seed)
        paths = list_echo_videos(data_root)
        if max_videos is not None:
            paths = paths[:max_videos]
        self.video_paths = paths
        self.augment = augment

    def __len__(self) -> int:
        return max(1, len(self.video_paths) * 200)

    def _augment(self, img1: np.ndarray, img2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # Random horizontal flip
        if self._rng.random() < 0.5:
            img1 = np.fliplr(img1).copy()
            img2 = np.fliplr(img2).copy()

        # Random rotation (small angle)
        angle = self._rng.uniform(-5, 5) # degrees
        h, w = self.image_hw
        center = (w / 2, h / 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        img1 = cv2.warpAffine(img1, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
        img2 = cv2.warpAffine(img2, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

        return img1, img2

    def _sample_pair(self) -> Tuple[np.ndarray, np.ndarray]:
        path = self._rng.choice(self.video_paths)
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {path}")
        try:
            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if n < 2:
                raise RuntimeError(f"Video has < 2 frames: {path}")
            i = self._rng.randrange(n)
            j = self._rng.randrange(n - 1)
            if j >= i:
                j += 1
        finally:
            cap.release()
        fi = _read_frame(path, i, self.image_hw)
        fj = _read_frame(path, j, self.image_hw)
        if fi is None or fj is None:
            raise RuntimeError(f"Failed to read frames from {path}")
        return fi, fj

    def __getitem__(self, index: int):
        if not self.video_paths:
            raise RuntimeError(
                f"No videos under {os.path.join(self.data_root, 'A4C', 'Videos')}. "
                "Mount your bucket or set DEMO = True."
            )
        moving, fixed = self._sample_pair()

        if self.augment:
            moving, fixed = self._augment(moving, fixed)

        src = torch.from_numpy(moving)[None, ...]
        tgt = torch.from_numpy(fixed)[None, ...]
        return src, tgt

In [66]:
class SyntheticPairDataset(Dataset):
    def __init__(self, n: int = 512, image_hw: Tuple[int, int] = (128, 128), seed: int = 0, augment: bool = False):
        super().__init__()
        self.n = n
        self.image_hw = image_hw
        self._rng = np.random.RandomState(seed)
        self.augment = augment

    def __len__(self) -> int:
        return self.n

    def _augment(self, img1: np.ndarray, img2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # Random horizontal flip
        if self._rng.rand() < 0.5:
            img1 = np.fliplr(img1).copy()
            img2 = np.fliplr(img2).copy()

        # Random rotation (small angle)
        angle = self._rng.uniform(-5, 5) # degrees
        h, w = self.image_hw
        center = (w / 2, h / 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        img1 = cv2.warpAffine(img1, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
        img2 = cv2.warpAffine(img2, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

        return img1, img2

    def _blob(self) -> np.ndarray:
        h, w = self.image_hw
        cx, cy = self._rng.rand(2) * 0.6 + 0.2
        sx, sy = self._rng.rand(2) * 0.15 + 0.05
        ys, xs = np.mgrid[0:1 : complex(0, h), 0:1 : complex(0, w)]
        g = np.exp(-(((xs - cx) / sx) ** 2 + ((ys - cy) / sy) ** 2))
        g = (g - g.min()) / (g.max() - g.min() + 1e-8)
        return g.astype(np.float32)

    def __getitem__(self, index: int):
        fixed = self._blob()
        dx, dy = self._rng.randn(2) * 3.0
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        warped = cv2.warpAffine(
            fixed, M, (self.image_hw[1], self.image_hw[0]), borderMode=cv2.BORDER_REFLECT
        )

        if self.augment:
            warped, fixed = self._augment(warped, fixed)

        src = torch.from_numpy(warped)[None, ...]
        tgt = torch.from_numpy(fixed)[None, ...]
        return src, tgt

In [67]:
def train():
    torch.manual_seed(SEED)
    hw = (IMAGE_SIZE, IMAGE_SIZE)

    if RUN_COLAB_SETUP:
        colab_setup_from_dataloading()

    resolved_root: Optional[str] = None
    if not DEMO:
        resolved_root = discover_data_root()
        if resolved_root is None:
            raise RuntimeError(
                "No EchoNet videos found under A4C/Videos.\n"
                "  • Run dataloading.py first (Drive mount + unzip + gcsfuse), or\n"
                "  • Set RUN_COLAB_SETUP = True (runs the same steps as dataloading.py), or\n"
                "  • Set DATA_ROOT to the folder that contains A4C/Videos, or ECHO_DATA_ROOT env, or\n"
                "  • Set DEMO = True for synthetic data.\n"
                "Expected layouts: /mnt/data/A4C/Videos (gcsfuse) or /content/local_data/A4C/Videos (unzip)."
            )

    if DEMO:
        ds = SyntheticPairDataset(n=50, image_hw=hw, seed=SEED, augment=True) # Reduced synthetic dataset size for sanity check
        eval_ds = SyntheticPairDataset(n=10, image_hw=hw, seed=SEED + 1) # Separate dataset for evaluation
        print("DEMO mode: synthetic pairs (no EchoNet videos).")
    else:
        assert resolved_root is not None
        ds = EchoFramePairDataset(
            data_root=resolved_root,
            image_hw=hw,
            seed=SEED,
            max_videos=MAX_VIDEOS,
            augment=True, # Enable augmentation for training data
        )
        # For real data, you'd typically split your video paths into train/val/test sets
        # For this example, we'll use a small subset of the training videos for eval
        # In a real scenario, ensure no overlap between training and evaluation videos
        eval_ds = EchoFramePairDataset(
            data_root=resolved_root,
            image_hw=hw,
            seed=SEED + 1,
            max_videos=min(1, MAX_VIDEOS) if MAX_VIDEOS else 1, # Use a tiny number of videos for eval
            augment=False, # No augmentation for evaluation data
        )
        nvid = len(ds.video_paths)
        vd = _videos_dir(resolved_root)
        print(f"Using data_root={resolved_root!r}  ({nvid} videos under {vd}) (limited to MAX_VIDEOS={MAX_VIDEOS})")
        if nvid == 0:
            raise RuntimeError(f"No video files found in {vd}")

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=BATCH_SIZE,
        shuffle=False, # No need to shuffle evaluation data
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )

    inshape = (IMAGE_SIZE, IMAGE_SIZE)
    model = VxmDense(
        inshape,
        int_steps=7,
        int_downsize=2,
        bidir=False,
    ).to(DEVICE)

    ncc = NCC()
    grad = Grad2D(penalty="l1")
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    save_dir = os.path.dirname(SAVE_PATH)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for epoch in range(EPOCHS):
        t0 = time.time()
        train_losses = []
        model.train()
        # Wrap the loader with tqdm for a progress bar
        for batch_idx, (src, tgt) in enumerate(tqdm(loader, desc=f"Epoch {epoch + 1}/{EPOCHS} Train")):
            src = src.to(DEVICE)
            tgt = tgt.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            y_source, pre_flow = model(src, tgt, registration=False)
            loss_sim = ncc.loss(tgt, y_source)
            loss_reg = grad.loss(pre_flow)
            loss = loss_sim + LAMBDA_GRAD * loss_reg
            loss.backward()
            opt.step()
            train_losses.append(loss.item())
            # Print batch loss every 100 batches (or adjust as needed)
            if (batch_idx + 1) % 100 == 0:
                print(f"  Batch {batch_idx + 1}/{len(loader)} Train Loss: {loss.item():.5f}")

        dt_train = time.time() - t0
        avg_train_loss = sum(train_losses) / len(train_losses)

        # Evaluation step
        model.eval()
        eval_losses = []
        with torch.no_grad(): # Disable gradient calculations for evaluation
            for batch_idx, (src_eval, tgt_eval) in enumerate(tqdm(eval_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} Eval ")):
                src_eval = src_eval.to(DEVICE)
                tgt_eval = tgt_eval.to(DEVICE)
                y_source_eval, pre_flow_eval = model(src_eval, tgt_eval, registration=False)
                loss_sim_eval = ncc.loss(tgt_eval, y_source_eval)
                loss_reg_eval = grad.loss(pre_flow_eval)
                eval_loss = loss_sim_eval + LAMBDA_GRAD * loss_reg_eval
                eval_losses.append(eval_loss.item())
        avg_eval_loss = sum(eval_losses) / len(eval_losses)

        print(
            f"epoch {epoch + 1}/{EPOCHS}  " +
            f"train_loss={avg_train_loss:.5f}  " +
            f"eval_loss={avg_eval_loss:.5f}  " +
            f"time={dt_train:.1f}s"
        )

    torch.save(
        {
            "model": model.state_dict(),
            "inshape": inshape,
            "config": {
                "DATA_ROOT": resolved_root if not DEMO else None,
                "IMAGE_SIZE": IMAGE_SIZE,
                "DEMO": DEMO,
            },
        },
        SAVE_PATH,
    )
    print(f"Saved {SAVE_PATH}")

In [70]:
if __name__ == "__main__":
    train()

Mounting Google Drive…
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping pediatric_echo_avi.zip to /content/local_data …
Authenticate for GCS (bucket access)…
Installing gcsfuse…
Mounting GCS bucket rayans_aimi_group_four_bucket to /mnt/data …
✅ Setup complete. Listing /mnt/data:
Using data_root='/mnt/data'  (50 videos under /mnt/data/A4C/Videos) (limited to MAX_VIDEOS=50)


Epoch 1/10 Train:   0%|          | 0/2500 [00:00<?, ?it/s]

  Batch 100/2500 Train Loss: -0.41328
  Batch 200/2500 Train Loss: -0.34320
  Batch 300/2500 Train Loss: -0.45446
  Batch 400/2500 Train Loss: -0.32011
  Batch 500/2500 Train Loss: -0.38891
  Batch 600/2500 Train Loss: -0.37899
  Batch 700/2500 Train Loss: -0.35658
  Batch 800/2500 Train Loss: -0.34785
  Batch 900/2500 Train Loss: -0.39668
  Batch 1000/2500 Train Loss: -0.40776
  Batch 1100/2500 Train Loss: -0.32571
  Batch 1200/2500 Train Loss: -0.36284
  Batch 1300/2500 Train Loss: -0.28773
  Batch 1400/2500 Train Loss: -0.29718
  Batch 1500/2500 Train Loss: -0.44175
  Batch 1600/2500 Train Loss: -0.38378
  Batch 1700/2500 Train Loss: -0.36802
  Batch 1800/2500 Train Loss: -0.42430
  Batch 1900/2500 Train Loss: -0.37643
  Batch 2000/2500 Train Loss: -0.33670
  Batch 2100/2500 Train Loss: -0.37167
  Batch 2200/2500 Train Loss: -0.34801
  Batch 2300/2500 Train Loss: -0.36410
  Batch 2400/2500 Train Loss: -0.34184
  Batch 2500/2500 Train Loss: -0.33063


Epoch 1/10 Eval :   0%|          | 0/50 [00:00<?, ?it/s]

epoch 1/10  train_loss=-0.34799  eval_loss=-0.40436  time=112.3s


...

epoch 9/10  train_loss=-0.38382  eval_loss=-0.43421  time=94.8s


Epoch 10/10 Train:   0%|          | 0/2500 [00:00<?, ?it/s]

  Batch 100/2500 Train Loss: -0.45693
  Batch 200/2500 Train Loss: -0.38156
  Batch 300/2500 Train Loss: -0.48284
  Batch 400/2500 Train Loss: -0.37762
  Batch 500/2500 Train Loss: -0.41770
  Batch 600/2500 Train Loss: -0.39923
  Batch 700/2500 Train Loss: -0.38756
  Batch 800/2500 Train Loss: -0.39690
  Batch 900/2500 Train Loss: -0.42176
  Batch 1000/2500 Train Loss: -0.43113
  Batch 1100/2500 Train Loss: -0.35580
  Batch 1200/2500 Train Loss: -0.41350
  Batch 1300/2500 Train Loss: -0.32182
  Batch 1400/2500 Train Loss: -0.33441
  Batch 1500/2500 Train Loss: -0.47677
  Batch 1600/2500 Train Loss: -0.42621
  Batch 1700/2500 Train Loss: -0.40200
  Batch 1800/2500 Train Loss: -0.45771
  Batch 1900/2500 Train Loss: -0.40586
  Batch 2000/2500 Train Loss: -0.37663
  Batch 2100/2500 Train Loss: -0.40509
  Batch 2200/2500 Train Loss: -0.40186
  Batch 2300/2500 Train Loss: -0.40344
  Batch 2400/2500 Train Loss: -0.38191
  Batch 2500/2500 Train Loss: -0.36375


Epoch 10/10 Eval :   0%|          | 0/50 [00:00<?, ?it/s]

epoch 10/10  train_loss=-0.38475  eval_loss=-0.43508  time=96.0s
Saved /content/vxm_echo.pt
